## Explore what samples were taken forward for sequencing, and prevalence by study

In the paper, we say:
```
A total of 2,614 participants were enrolled in the household survey, and 6,858 participants with malaria symptoms in the health facility-based study. Based on rapid diagnostic test and PCR positivity, we sequenced 2,399 (25%) of the 9,472 isolates collected. Of these, 449 (19%) were excluded due to low sequencing coverage, evidence of sequencing contamination, or evidence of sample duplication. For the remaining 1,950 isolates, the genotype of Pfkelch13 was assessed (Table S2).
```

In [1]:
import os
import pandas as pd
import geopandas as gpd
import seaborn as sns

import statsmodels.formula.api as smf
import statsmodels.api as sm
import numpy as np

import matplotlib.pyplot as plt
import matplotlib as mpl
from matplotlib.lines import Line2D
from matplotlib.gridspec import GridSpec

In [2]:
# Global font size parameters -- aim to produce plots at correct size
mpl.rcParams['font.size'] = 8  # base font size
# mpl.rcParams['axes.titlesize'] = 16
# mpl.rcParams['axes.labelsize'] = 14
# mpl.rcParams['xtick.labelsize'] = 12
# mpl.rcParams['ytick.labelsize'] = 12
# mpl.rcParams['legend.fontsize'] = 12
# mpl.rcParams['figure.titlesize'] = 18

## Settings

In [3]:
DIR_SEQDATA = "../seqdata/all/summaries/HRP23_MIS2024"
DIR_GEODATA = "../geodata/"
DIR_FIGS = "../figures/response/sample-selection"
os.makedirs(DIR_FIGS, exist_ok=True)
save_results = True

In [4]:
W = 3.4 # roughly width of one column
KEEP_GEOCOLUMNS = ['province', 'district', 'area_sqkm', 'center_lat', 'center_lon', 'geometry']

## Load data

In [5]:
## Geodata
df_geo = gpd.read_file(
    f"{DIR_GEODATA}/zmb_admin_boundaries.shp",
    layer="zmb_admin1"
)
df_geo['province'] = [s.lower().replace("-", "") for s in df_geo['adm1_name']]
df_geo = df_geo[[c for c in KEEP_GEOCOLUMNS if c != 'district']]
order_province = df_geo.sort_values("center_lon")["province"].str.capitalize().tolist()
order_study = ['MIS2024', 'HRP23', 'Both'] # by size

In [6]:
# dataframe containing all metadata, NOT duplicates
df_metadata = (pd.read_csv(
    "../metadata/2026-04-05_sample-duplication/table.zambia_all_data_combined.dedup.csv",
    dtype=dict(sample_id=str))
    .drop(['ward', 'healthfac',
           'lat', 'long', 'collection_date',
           'extraction_id', 'location'], axis=1)
)

**Compare count against duplicates**

In [7]:
_df_metadata = pd.read_csv(
    "../metadata/2026-01-23_consolidated-metadata/table.zambia_all_data_combined.csv"
)

In [8]:
_df_metadata.shape[0], df_metadata.shape[0], _df_metadata.shape[0] - df_metadata.shape[0]

(9528, 9472, 56)

In [9]:
# dataframe containing information about sample QC (coverage, contamination)
_df_coverage = (
    pd.read_csv("../seqdata/all/summaries/HRP23_MIS2024/summary.coverage.csv",
                dtype=dict(sample_id=str))
    .query("sample_type == 'field'")
    .query("name == 'kelch13-p383-727'")
    #.query("status == 'pass'")
)
print(f"Total: {_df_coverage.shape[0]} | Unique: {_df_coverage.sample_id.unique().shape[0] }")

# Load sequencing coverage data, and remove the duplicates
not_duplicated = [not 'duplicate' in status for status in _df_coverage.status]
df_coverage = _df_coverage.loc[not_duplicated]

Total: 2720 | Unique: 2399


### Merge and determine outcome of each sample

In [10]:
df_merge_inclusion = pd.merge(
    left=df_metadata,
    right=df_coverage[['sample_id', 'passing', 'status']],
    on='sample_id',
    how='left',
    validate='1:1'
)

In [11]:
# Bin and clean parasitemia
bin_parasitemia = [10**i for i in [-1, 0, 1, 2, 3, 4, 10]]
df_merge_inclusion['parasitemia_bins'] = (pd.cut(df_merge_inclusion['parasitemia'], bin_parasitemia)
    .cat.add_categories(['Missing'])
    .fillna('Missing')
)
df_merge_inclusion['parasitemia_log10'] = np.log10(df_merge_inclusion['parasitemia'])

In [12]:
# Create an outcome for each sample in the study
# - this collapses the breakdown from `nomadic summarize` into key categories
df_merge_inclusion['outcome'] = (
    ['failed' if (status != 'pass' and status != 'not_sequenced') else status
     for status in df_merge_inclusion['status'].fillna('not_sequenced')]
)

In [13]:
df_merge_inclusion['status'].value_counts()

status
pass             1950
lowcov            382
contam             49
contam;lowcov      18
Name: count, dtype: int64

In [14]:
449 / 2399

0.18716131721550647

In [15]:
400 / 2399

0.16673614005835766

In [16]:
49 + 18

67

In [17]:
df_merge_inclusion['outcome'].value_counts()

outcome
not_sequenced    7073
pass             1950
failed            449
Name: count, dtype: int64

In [18]:
df_merge_inclusion.insert(0, 'All', 'All')

In [19]:
df_all = (df_merge_inclusion.groupby(['All'])
               .agg(
                   N=pd.NamedAgg('outcome', len),
                   n_seq=pd.NamedAgg('outcome', lambda x: (x != 'not_sequenced').sum()),
                   n_pass=pd.NamedAgg('status', lambda x: (x == 'pass').sum())
               )
               .assign(
                   per_seq=lambda df: 100 * df.n_seq / df.N,
                   per_pass_of_seq=lambda df: 100 * df.n_pass / df.n_seq,
                   per_pass_of_N=lambda df: 100 * df.n_pass / df.N,
                   clean_seq=lambda df: [f"{n} ({p:.1f})" for n, p in zip(df.n_seq, df.per_seq)],
                   clean_pass_seq=lambda df: [f"{n} ({p:.1f})" for n, p in zip(df.n_pass, df.per_pass_of_seq)],
                   clean_pass_N=lambda df: [f"{n} ({p:.1f})" for n, p in zip(df.n_pass, df.per_pass_of_N)],
               )
               .reset_index()
              )
df_all.insert(0, 'study', 'Both')
df_all.drop('All', axis=1, inplace=True)

In [20]:
df_study = (df_merge_inclusion.groupby(['study'])
               .agg(
                   N=pd.NamedAgg('outcome', len),
                   n_seq=pd.NamedAgg('outcome', lambda x: (x != 'not_sequenced').sum()),
                   n_pass=pd.NamedAgg('status', lambda x: (x == 'pass').sum())
               )
               .assign(
                   per_seq=lambda df: 100 * df.n_seq / df.N,
                   per_pass_of_seq=lambda df: 100 * df.n_pass / df.n_seq,
                   per_pass_of_N=lambda df: 100 * df.n_pass / df.N,
                   clean_seq=lambda df: [f"{n} ({p:.1f})" for n, p in zip(df.n_seq, df.per_seq)],
                   clean_pass_seq=lambda df: [f"{n} ({p:.1f})" for n, p in zip(df.n_pass, df.per_pass_of_seq)],
                   clean_pass_N=lambda df: [f"{n} ({p:.1f})" for n, p in zip(df.n_pass, df.per_pass_of_N)],
               )
               .reset_index()
              )

In [21]:
columns_all = {
    "study": "Study",
    "N": "Collected - n",
    "clean_seq": "Sequenced - n (% collected)",
    "clean_pass_seq": "Passed QC - n (% sequenced)",
    "clean_pass_N": "Passed QC - n (% collected)",
}

In [22]:
df_clean_study = pd.concat([df_study, df_all])[
    columns_all.keys()
].rename(columns_all, axis=1)
df_clean_study['Study'] = pd.Categorical(df_clean_study['Study'],
                                         categories=order_study, ordered=True)
df_clean_study = df_clean_study.sort_values("Study")
df_clean_study

,Study,Collected - n,Sequenced - n (% collected),Passed QC - n (% sequenced),Passed QC - n (% collected)
1,MIS2024,2614,435 (16.6),283 (65.1),283 (10.8)
0,HRP23,6858,1964 (28.6),1667 (84.9),1667 (24.3)
0,Both,9472,2399 (25.3),1950 (81.3),1950 (20.6)


### By province

In [23]:
columns_province = {
    "study": "Study",
    "province": "Province",
    "N": "Collected - n",
    "clean_seq": "Sequenced - n (% collected)",
    "clean_pass_seq": "Passed QC - n (% sequenced)",
    "clean_pass_N": "Passed QC - n (% collected)",
}

In [24]:
df_province = (df_merge_inclusion.groupby(['province'])
               .agg(
                   N=pd.NamedAgg('outcome', len),
                   n_seq=pd.NamedAgg('outcome', lambda x: (x != 'not_sequenced').sum()),
                   n_pass=pd.NamedAgg('status', lambda x: (x == 'pass').sum())
               )
               .assign(
                   per_seq=lambda df: 100 * df.n_seq / df.N,
                   per_pass_of_seq=lambda df: 100 * df.n_pass / df.n_seq,
                   per_pass_of_N=lambda df: 100 * df.n_pass / df.N,
                   clean_seq=lambda df: [f"{n} ({p:.1f})" for n, p in zip(df.n_seq, df.per_seq)],
                   clean_pass_seq=lambda df: [f"{n} ({p:.1f})" for n, p in zip(df.n_pass, df.per_pass_of_seq)],
                   clean_pass_N=lambda df: [f"{n} ({p:.1f})" for n, p in zip(df.n_pass, df.per_pass_of_N)],
               )
               .reset_index()
              )
df_province.insert(0, 'study', 'Both')

In [25]:
df_study_province = (df_merge_inclusion.groupby(['study', 'province'])
               .agg(
                   N=pd.NamedAgg('outcome', len),
                   n_seq=pd.NamedAgg('outcome', lambda x: (x != 'not_sequenced').sum()),
                   n_pass=pd.NamedAgg('status', lambda x: (x == 'pass').sum())
               )
               .assign(
                   per_seq=lambda df: 100 * df.n_seq / df.N,
                   per_pass_of_seq=lambda df: 100 * df.n_pass / df.n_seq,
                   per_pass_of_N=lambda df: 100 * df.n_pass / df.N,
                   clean_seq=lambda df: [f"{n} ({p:.1f})" for n, p in zip(df.n_seq, df.per_seq)],
                   clean_pass_seq=lambda df: [f"{n} ({p:.1f})" for n, p in zip(df.n_pass, df.per_pass_of_seq)],
                   clean_pass_N=lambda df: [f"{n} ({p:.1f})" for n, p in zip(df.n_pass, df.per_pass_of_N)],
               )
               .reset_index()
              )

In [26]:
df_clean_province = pd.concat([df_study_province, df_province, df_study, df_all])[
    columns_province.keys()
    ].rename(columns_province, axis=1)
df_clean_province.Province = df_clean_province.Province.str.capitalize()

In [27]:
# Prepare for ordering
df_clean_province['Study'] = pd.Categorical(df_clean_province['Study'],
                                            categories=order_study, ordered=True)
df_clean_province['Province'] = pd.Categorical(df_clean_province['Province'].str.capitalize(), 
                                               categories=order_province, ordered=True)
df_clean_province = df_clean_province.sort_values(['Study', 'Province'])

In [28]:
df_clean_province

,Study,Province,Collected - n,Sequenced - n (% collected),Passed QC - n (% sequenced),Passed QC - n (% collected)
18,MIS2024,Western,186,44 (23.7),38 (86.4),38 (20.4)
16,MIS2024,Northwestern,116,51 (44.0),38 (74.5),38 (32.8)
17,MIS2024,Southern,300,1 (0.3),1 (100.0),1 (0.3)
10,MIS2024,Copperbelt,329,47 (14.3),37 (78.7),37 (11.2)
12,MIS2024,Luapula,230,86 (37.4),29 (33.7),29 (12.6)
13,MIS2024,Lusaka,305,15 (4.9),8 (53.3),8 (2.6)
9,MIS2024,Central,248,30 (12.1),19 (63.3),19 (7.7)
15,MIS2024,Northern,187,64 (34.2),48 (75.0),48 (25.7)
14,MIS2024,Muchinga,119,38 (31.9),31 (81.6),31 (26.1)
11,MIS2024,Eastern,594,59 (9.9),34 (57.6),34 (5.7)


## By parasitemia

In [29]:
columns_parasitemia = {
    "study": "Study",
    "parasitemia_bins": 'Parasite Density',
    "N": "Collected - n",
    "clean_seq": "Sequenced - n (% collected)",
    "clean_pass_seq": "Passed QC - n (% sequenced)",
    "clean_pass_N": "Passed QC - n (% collected)",
}

In [30]:
df_parasitemia = (df_merge_inclusion.groupby(['parasitemia_bins'], observed=True)
               .agg(
                   N=pd.NamedAgg('outcome', len),
                   n_seq=pd.NamedAgg('outcome', lambda x: (x != 'not_sequenced').sum()),
                   n_pass=pd.NamedAgg('status', lambda x: (x == 'pass').sum())
               )
               .assign(
                   per_seq=lambda df: 100 * df.n_seq / df.N,
                   per_pass_of_seq=lambda df: 100 * df.n_pass / df.n_seq,
                   per_pass_of_N=lambda df: 100 * df.n_pass / df.N,
                   clean_seq=lambda df: [f"{n} ({p:.1f})" for n, p in zip(df.n_seq, df.per_seq)],
                   clean_pass_seq=lambda df: [f"{n} ({p:.1f})" for n, p in zip(df.n_pass, df.per_pass_of_seq)],
                   clean_pass_N=lambda df: [f"{n} ({p:.1f})" for n, p in zip(df.n_pass, df.per_pass_of_N)],
               )
               .reset_index()
              )
df_parasitemia.insert(0, 'study', 'Both')

In [31]:
df_study_parasitemia = (df_merge_inclusion.groupby(['study', 'parasitemia_bins'], observed=True)
               .agg(
                   N=pd.NamedAgg('outcome', len),
                   n_seq=pd.NamedAgg('outcome', lambda x: (x != 'not_sequenced').sum()),
                   n_pass=pd.NamedAgg('status', lambda x: (x == 'pass').sum())
               )
               .assign(
                   per_seq=lambda df: 100 * df.n_seq / df.N,
                   per_pass_of_seq=lambda df: 100 * df.n_pass / df.n_seq,
                   per_pass_of_N=lambda df: 100 * df.n_pass / df.N,
                   clean_seq=lambda df: [f"{n} ({p:.1f})" for n, p in zip(df.n_seq, df.per_seq)],
                   clean_pass_seq=lambda df: [f"{n} ({p:.1f})" for n, p in zip(df.n_pass, df.per_pass_of_seq)],
                   clean_pass_N=lambda df: [f"{n} ({p:.1f})" for n, p in zip(df.n_pass, df.per_pass_of_N)],
               )
               .reset_index()
              )

In [32]:
df_clean_parasitemia = pd.concat([df_study_parasitemia, df_parasitemia, df_study, df_all])[
    columns_parasitemia.keys()
    ].rename(columns_parasitemia, axis=1)
df_clean_parasitemia['Study'] = pd.Categorical(df_clean_parasitemia['Study'],
                                               categories=order_study, ordered=True)
df_clean_parasitemia = df_clean_parasitemia.sort_values(["Study", 'Parasite Density'])

In [33]:
df_clean_parasitemia

,Study,Parasite Density,Collected - n,Sequenced - n (% collected),Passed QC - n (% sequenced),Passed QC - n (% collected)
5,MIS2024,"(10.0, 100.0]",20,9 (45.0),7 (77.8),7 (35.0)
6,MIS2024,"(100.0, 1000.0]",104,81 (77.9),53 (65.4),53 (51.0)
7,MIS2024,"(1000.0, 10000.0]",120,108 (90.0),88 (81.5),88 (73.3)
8,MIS2024,"(10000.0, 10000000000.0]",113,97 (85.8),75 (77.3),75 (66.4)
9,MIS2024,Missing,2257,140 (6.2),60 (42.9),60 (2.7)
1,MIS2024,NaN,2614,435 (16.6),283 (65.1),283 (10.8)
0,HRP23,"(10.0, 100.0]",108,42 (38.9),28 (66.7),28 (25.9)
1,HRP23,"(100.0, 1000.0]",452,206 (45.6),158 (76.7),158 (35.0)
2,HRP23,"(1000.0, 10000.0]",509,313 (61.5),296 (94.6),296 (58.2)
3,HRP23,"(10000.0, 10000000000.0]",821,618 (75.3),608 (98.4),608 (74.1)


## Write tables

In [34]:
if save_results:
    df_clean_study.to_excel(f"{DIR_FIGS}/table.samples_by_study.xlsx", index=False)
    df_clean_province.to_excel(f"{DIR_FIGS}/table.samples_by_province.xlsx", index=False)
    df_clean_parasitemia.to_excel(f"{DIR_FIGS}/table.samples_by_parasitemia.xlsx", index=False)

In [35]:
df_merge_inclusion.query("province == 'southern'").rdt_result.value_counts()

rdt_result
pf_neg    297
pf_pos      3
Name: count, dtype: int64

In [36]:
df_merge_inclusion.query("province == 'southern'").has_pcr.sum()

np.int64(2)

In [37]:
df_merge_inclusion.query("province == 'southern'").take_for_sequencing.sum()

1

In [38]:
df_merge_inclusion.query("province == 'southern'").query("rdt_result == 'pf_pos'")

,All,study,sample_id,province,district,rdt_result,parasitemia,take_for_sequencing,screening_method,has_pcr,passing,status,parasitemia_bins,parasitemia_log10,outcome
1632,All,MIS2024,8030015,southern,namwala,pf_pos,NaN,NaN,NaN,False,NaN,NaN,Missing,NaN,not_sequenced
1633,All,MIS2024,8030016,southern,namwala,pf_pos,NaN,False,pet_pcr,True,NaN,NaN,Missing,NaN,not_sequenced
2528,All,MIS2024,8033071,southern,siavonga,pf_pos,105.0,True,pet_pcr,True,True,pass,"(100.0, 1000.0]",2.021189,pass
